# 09 - TFLite Micro Compatibility Validation

**Objective:** Cross-check every operator in the final INT8 `.tflite` model against the TFLite Micro supported-ops list, and validate the quantized model's predictions against the desktop `tf.lite.Interpreter` as a stand-in for an embedded TFLite Micro runtime (no physical embedded runtime is available in this environment).

In [1]:
import sys, os
from pathlib import Path
ML_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ML_ROOT / 'preprocessing'))
sys.path.insert(0, str(ML_ROOT / 'scripts'))
sys.path.insert(0, str(ML_ROOT / 'models'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
FIG_DIR = ML_ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
print('ML_ROOT =', ML_ROOT)


ML_ROOT = /Users/kireeti/Desktop/Projects/RESQ/SECURE-FOREST-PATROL/ml


In [2]:
import tensorflow as tf, json
TFLM_SUPPORTED_OPS = {'CONV_2D','DEPTHWISE_CONV_2D','FULLY_CONNECTED','AVERAGE_POOL_2D',
    'MAX_POOL_2D','SOFTMAX','RESHAPE','QUANTIZE','DEQUANTIZE','MEAN','ADD','MUL','RELU',
    'RELU6','PAD','CONCATENATION','LOGISTIC'}
interp = tf.lite.Interpreter(
    model_path=str(ML_ROOT/'models'/'final'/'forest_acoustic_int8.tflite'),
    experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES)
interp.allocate_tensors()
ops_used = {op['op_name'] for op in interp._get_ops_details()}
unsupported = ops_used - TFLM_SUPPORTED_OPS
print('Ops used:', sorted(ops_used))
print('Unsupported TFLM ops:', sorted(unsupported) if unsupported else 'NONE - all ops supported')

Ops used: ['CONV_2D', 'DEPTHWISE_CONV_2D', 'FULLY_CONNECTED', 'MAX_POOL_2D', 'MEAN', 'SOFTMAX']
Unsupported TFLM ops: NONE - all ops supported


/Users/kireeti/Desktop/Projects/RESQ/SECURE-FOREST-PATROL/ml/venv/lib/python3.10/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


## Prediction-consistency check against Keras FP32 (test set)

Since no physical ESP32-S3 is available, `tf.lite.Interpreter` running the exact INT8 operator graph is used as the stand-in embedded-runtime validator.

In [3]:
import numpy as np
sys.path.insert(0, str(ML_ROOT/'models'))
from train_cnn import load_split
final_meta = json.load(open(ML_ROOT/'models'/'final'/'final_metadata.json'))
mean, std = final_meta['feature_mean'], final_meta['feature_std']
mel_test, y_test = load_split('test')
X_test = ((mel_test-mean)/std)[..., np.newaxis].astype(np.float32)
keras_model = tf.keras.models.load_model(ML_ROOT/'models'/'final'/'final.keras')
keras_pred = np.argmax(keras_model.predict(X_test, verbose=0), axis=1)
in_d = interp.get_input_details()[0]; out_d = interp.get_output_details()[0]
scale, zero = in_d['quantization']
int8_pred = []
for i in range(len(X_test)):
    xq = np.round(X_test[i:i+1]/scale + zero).astype(np.int8)
    interp.set_tensor(in_d['index'], xq)
    interp.invoke()
    int8_pred.append(np.argmax(interp.get_tensor(out_d['index'])[0]))
int8_pred = np.array(int8_pred)
agreement = (keras_pred == int8_pred).mean()
print(f'Keras FP32 vs INT8-TFLite-Interpreter prediction agreement: {agreement:.4f}')

Keras FP32 vs INT8-TFLite-Interpreter prediction agreement: 0.9821


## Conclusion

All operators in the final INT8 model (`CONV_2D`, `DEPTHWISE_CONV_2D`, `FULLY_CONNECTED`, `MAX_POOL_2D`, `MEAN`, `SOFTMAX`) are in the TFLite Micro commonly-supported op set — no architecture changes were needed. Desktop-interpreter validation shows high agreement with the FP32 model, but this is **not** a substitute for running on actual TFLite Micro / ESP32-S3 hardware, which was not available in this environment (see 'HARDWARE DEPLOYMENT BLOCKED' in `reports/FINAL_MODEL_REPORT.md`).